# Step 1 Phase A-10：W₂ 推定量・orientation-cluster m 感度・global 較正経路 **v1.0**
2026-09-12。研究計画 v0.5（定義3・定義4・§6）と A10 設計ノート v1.0 を実装する Phase A 調査ノートブック。凍結資産のみを入力とする（A5 B-stack・A11 cov_cache の E7_b1_A x₀⁽¹⁾・Step 0 CVEC/観測値・t2b2_bridge・A9/A11 の D(R) 求積構成）。
**A10-2**：1R:m z の m 感度（m=10 vs 100・各 2 独立反復・N=10⁶・PR3-power-matched 系統・ℓ2–4 float64 評価器・paired cluster bootstrap）。判定：|logQ(100)−logQ(10)| ≤ 1.96·√(se₁₀²+se₁₀₀²) かつ m=100 の 2 反復が精度 gate 通過なら m=100。
**A10-3**：global false-support 較正の経路検証（再走査なし：T₁ 閾値の indicator 再評価＋cluster 集計表 bootstrap・n_pseudo=200・Wilson 上側 CI）。1 モデル点・1 系統の経路検証であり，較正そのものではない。
**A10-1**：実エンジン出力での exact 2D W₂（POT emd2・n_sub=5000・cluster 単位部分標本・校正サンプルで whitening・null B=200・q99）と W₂(model, iso)。
gate：資産 SHA 9・共分散/平方根 2・D(R) 4・走査 1・**等方エンジン vs A5 map-based null の整合**（中央値が A5 bootstrap 95% CI 内）・W₂ 有限・inventory exact＝19。Phase A 調査段階：ここでの数値は rules v1.0 draft の設計根拠であり凍結成果物ではない。


In [ ]:
# ---- 1. mode / environment / pinned mirror-topology / output ----
import os, sys, json, hashlib, subprocess, platform
A10_MODE = globals().get('A10_MODE', 'official')                      # add a cell `A10_MODE='smoke'` above for the smoke run
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
MT_COMMIT = '1bdd9ea8a00891d6dc3622331f6dad5b86c16c89'                  # A8 freeze commit (contains A5/A9/A11/A8 freezes, t1_engine, t2b2_bridge, Step 0)
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
    BASE = '/content/drive/MyDrive/mirror_topology'; WORK = '/content/a10_work'; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pot'], check=True)
else:
    BASE = os.environ.get('A10_BASE', '/tmp/a10w/base'); WORK = os.environ.get('A10_WORK', '/tmp/a10w')
os.makedirs(WORK, exist_ok=True); MT = os.path.join(WORK, 'mt_a10')
if not os.path.exists(os.path.join(MT, '.git')): subprocess.run(['git', 'clone', '-q', 'https://github.com/tsujikeita/mirror-topology.git', MT], check=True)
subprocess.run(['git', '-C', MT, 'fetch', '-q', 'origin', MT_COMMIT], check=False); subprocess.run(['git', '-C', MT, 'checkout', '-q', MT_COMMIT], check=True)
assert subprocess.run(['git', '-C', MT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() == MT_COMMIT
OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a10_v1.0_{A10_MODE}'); os.makedirs(OUT, exist_ok=True)
os.environ.update(A10_MODE=A10_MODE, A10_MT=MT, A10_OUT=OUT, A10_MT_COMMIT=MT_COMMIT)
# notebook identity (source-only SHA of the committed copy vs this live source, official only)
sys.path.insert(0, MT); import t2b2_run as tr
NB_BASENAME = 'MirrorTopology_Step1_A10_v1.0.ipynb'; ident = dict(basename=NB_BASENAME)
try:
    ident['head_copy'] = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except Exception as e: ident['head_copy'] = None
os.environ['A10_NB_IDENTITY'] = json.dumps(ident)
import numpy, scipy, healpy, pandas
VERS = dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, healpy=healpy.__version__, pandas=pandas.__version__)
EXPECTED = dict(python='3.13.15', numpy='2.1.3', scipy='1.16.3', healpy='1.20.0')   # Colab official environment (A8/A11 official)
if A10_MODE == 'official' and IN_COLAB:
    mism = {k: (VERS[k], v) for k, v in EXPECTED.items() if VERS[k] != v}
    if mism: print('WARNING: version mismatch vs expected official environment (recorded in provenance):', mism)
print('mode', A10_MODE, '| MT', MT_COMMIT[:12], '| OUT', OUT, '| versions', VERS)


In [ ]:
import os, sys, json, time, hashlib, platform, warnings
import numpy as np, scipy, healpy as hp, pandas as pd
from scipy.spatial.transform import Rotation
from scipy.special import sph_harm_y
from scipy.stats import gaussian_kde
warnings.filterwarnings('ignore')
MODE = os.environ['A10_MODE']; MT = os.environ['A10_MT']; OUT = os.environ['A10_OUT']; os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, MT); import t1_engine as t1, t2b2_bridge as br
def sha(p): return hashlib.sha256(open(p, 'rb').read()).hexdigest()
def asha(a): return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()
GATES, DIAG, REC = {}, {}, {}; T0 = time.time()
CFG = dict(smoke=dict(N=20_000, N_CAL=20_000, N_PSEUDO=50, B_BOOT=500, B_NULL=10, N_SUB_W2=2000, N_FIT=5000),
           official=dict(N=1_000_000, N_CAL=200_000, N_PSEUDO=200, B_BOOT=2000, B_NULL=200, N_SUB_W2=5000, N_FIT=20_000))[MODE]
M_LIST = (10, 100); CHUNK = 10_000; MASTER_SEED = 20260911
STREAM = dict(gaussian=0, rotation=1, calibration=2, pseudo=3, bootstrap=4, w2=5)
def rng_for(stream, *ids): return np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 10, STREAM[stream], *[int(i) for i in ids]]))
# ---------------- frozen assets (SHA gates) ----------------
BST = os.path.join(MT, 'results/step1_phaseA/A5_freeze/s1_Bstack_l2_4_N16_common_v1.npz'); COVF = os.path.join(MT, 'results/step1_phaseA/A11_freeze/official/cov_cache/cov_E7_99337bfd75deea5fe.npy')
S0F = os.path.join(MT, 'docs/step0_frozen_Bpm_v1.npz'); S0CSV = os.path.join(MT, 'results/step0_v0.7/step0_official_v0_7.csv')
zB = np.load(BST); Bp, Bm = np.asarray(zB['Bp_stack'], np.float64), np.asarray(zB['Bm_stack'], np.float64)
GATES['G_bstack_file_sha'] = (sha(BST) == 'ec2d3eb501c3e00af85a505d95d7141fddb4ea23ab1da971906a5c21f80eef5f')
GATES['G_bstack_array_sha'] = (hashlib.sha256(np.ascontiguousarray(Bp).tobytes() + np.ascontiguousarray(Bm).tobytes()).hexdigest() == 'eb51414885785b77e9d3f7fbb25e1c9396f52e19c053d58113d50e206353a93f')
covman = json.load(open(COVF + '.manifest.json')); GATES['G_cov_manifest_target'] = (covman['manifest']['topology'] == 'E7' and covman['manifest']['params'] == dict(LAx=1.0, LAy=0.3, L1y=1.0, L2x=0.0, L2z=1.0) and covman['manifest']['x0'] == [0.31, 0.21, 0.42])
GATES['G_cov_file_sha'] = (sha(COVF) == covman['cov_file_sha256'] == '27e2bb589722526a40ddeca1b738f6c58edb1c8efd39672b0f7080d8de133308')
Mx, C_CT, covmeta = t1.load_cov_full(COVF, 4); GATES['G_cov_array_sha'] = (covmeta['cov_array_sha256'] == covman['cov_array_sha256'])
GATES['G_bridge_sha'] = (sha(os.path.join(MT, 't2b2_bridge.py')) == '45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872')
z0 = np.load(S0F, allow_pickle=True); CVEC = np.asarray(z0['CVEC'], np.float64); GATES['G_cvec_sha'] = (asha(CVEC) == '17d85b41ee0665d88418ebf8dee794d0da0a6f219053e983ec9b0501d763c8c6')
RB = br.real_basis_lm(); GATES['G_basis_order'] = ([tuple(b) for b in z0['basis_lm']] == [(int(l), int(m), cs) for (l, m, cs) in RB])
s0 = pd.read_csv(S0CSV); r0 = s0[s0['map'] == 'PR3_Commander']; T1o, T2o = 39.67178834527284, 259.3375006282747
GATES['G_step0_obs_bound'] = (len(r0) == 1 and float(r0.iloc[0]['Splus']) == T1o and float(r0.iloc[0]['Sminus']) == T2o)
# ---------------- covariance systems ----------------
LBLK = [slice(0, 5), slice(5, 12), slice(12, 21)]; c_ct = np.array([np.trace(C_CT[b, b]) / (2 * l + 1) for b, l in zip(LBLK, (2, 3, 4))]); c_pr3 = CVEC[[0, 5, 12]]
Dm = np.diag(np.concatenate([np.repeat(np.sqrt(c_pr3[i] / c_ct[i]), 2 * l + 1) for i, l in enumerate((2, 3, 4))])); C_MATCHED = Dm @ C_CT @ Dm; C_ISO = np.diag(CVEC)
GATES['G_matched_power'] = bool(np.allclose([np.trace(C_MATCHED[b, b]) / (2 * l + 1) for b, l in zip(LBLK, (2, 3, 4))], c_pr3, rtol=1e-12))
def psqrt(C):
    w, V = np.linalg.eigh(C); lam_min = float(w.min()); wc = np.where(w < 1e-12 * w.max(), 0.0, w); S = V @ np.diag(np.sqrt(wc)) @ V.T
    return S, dict(lambda_min_raw=lam_min, clip=float(np.sum(w < 1e-12 * w.max())), sym=float(np.linalg.norm(S - S.T) / np.linalg.norm(S)), recon=float(np.linalg.norm(S @ S.T - C) / np.linalg.norm(C)))
S_M, iM = psqrt(C_MATCHED); S_I, iI = psqrt(C_ISO); GATES['G_sqrt_hard'] = (iM['sym'] < 1e-12 and iM['recon'] < 1e-10 and iI['sym'] < 1e-12 and iI['recon'] < 1e-10 and iM['clip'] == 0)
REC['covariance'] = dict(system='PR3-power-matched (morphology-only covariance experiment)', c_l_CT=c_ct.tolist(), c_l_PR3=c_pr3.tolist(), scale=np.sqrt(c_pr3 / c_ct).tolist(), C_matched_sha256=asha(C_MATCHED), C_iso_sha256=asha(C_ISO), S_M_sha256=asha(S_M), S_I_sha256=asha(S_I), sqrt_model=iM, sqrt_iso=iI, ct_intake=covmeta['eig'])
# ---------------- rotation representation D(R) on the frozen real basis (A9/A11 quadrature construction) ----------------
LM = br.lm_full(); M21 = br.M_matrix()[0]; LMAX = 4; _NT = _NP = 2 * LMAX + 2
_xg, _wg = np.polynomial.legendre.leggauss(_NT); _th = np.arccos(_xg); _ph = 2 * np.pi * np.arange(_NP) / _NP
TH, PH = np.meshgrid(_th, _ph, indexing='ij'); WQ = (np.repeat(_wg[:, None], _NP, axis=1) * (2 * np.pi / _NP)).ravel()
DIRS = np.column_stack([np.sin(TH).ravel() * np.cos(PH).ravel(), np.sin(TH).ravel() * np.sin(PH).ravel(), np.cos(TH).ravel()])
def Yc_at(dirs):
    th, ph = hp.vec2ang(dirs); return np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM])
def Ymat(dirs): return (M21.conj() @ Yc_at(dirs)).real.T
YQ = Ymat(DIRS); YQW = (YQ * WQ[:, None]).T; GATES['G_quadrature_orthonormal'] = bool(np.abs(YQW @ YQ - np.eye(21)).max() < 1e-12)
def D_of_R(Rm): return YQW @ Ymat(DIRS @ Rm)
def D_batch(Rs):                                                     # vectorised D for a batch of rotations (K,3,3) -> (K,21,21)
    P = np.einsum('qj,kji->kqi', DIRS, Rs); th, ph = hp.vec2ang(P.reshape(-1, 3)); Y = np.array([sph_harm_y(l, m, th, ph) for (l, m) in LM]).reshape(21, len(Rs), -1)
    Yr = np.einsum('ab,bkq->kqa', M21.conj(), Y).real; return np.einsum('aq,kqb->kab', YQW, Yr)
rt = rng_for('rotation', 99); Rt = Rotation.random(num=8, rng=rt).as_matrix(); Db = D_batch(Rt)
GATES['G_D_batch_matches_single'] = bool(max(np.abs(Db[k] - D_of_R(Rt[k])).max() for k in range(8)) < 1e-12)
GATES['G_D_orthogonal'] = bool(np.abs(np.einsum('kab,kac->kbc', Db, Db) - np.eye(21)).max() < 1e-10)
GATES['G_D_homomorphism'] = bool(np.abs(D_of_R(Rt[0] @ Rt[1]) - Db[0] @ Db[1]).max() < 1e-10)
# ---------------- scan (A8 l24_feature231 route, float64) ----------------
iu = np.triu_indices(21); W = np.where(iu[0] == iu[1], 1.0, 2.0); FB_p = (Bp[:, iu[0], iu[1]] * W).T; FB_m = (Bm[:, iu[0], iu[1]] * W).T   # (231, 3072)
def scan(X):
    """X (n,21) -> T1 (min over axes of S+), axis, T2 (S- at that axis). float64, chunked."""
    n = len(X); T1 = np.empty(n); AX = np.empty(n, np.int32); T2 = np.empty(n)
    for a in range(0, n, CHUNK):
        f = X[a:a + CHUNK][:, iu[0]] * X[a:a + CHUNK][:, iu[1]]; S = f @ FB_p; ax = np.argmin(S, axis=1); AX[a:a + CHUNK] = ax; T1[a:a + CHUNK] = S[np.arange(len(ax)), ax]; T2[a:a + CHUNK] = np.einsum('ij,ji->i', f, FB_m[:, ax])
    return T1, AX, T2
xt = rng_for('gaussian', 999).standard_normal((3, 21)); t1d, axd, t2d = scan(xt)
GATES['G_scan_vs_direct'] = bool(max(abs(t1d[i] - min(xt[i] @ Bp[a] @ xt[i] for a in range(3072))) / abs(t1d[i]) for i in range(3)) < 1e-12 and max(abs(t2d[i] - xt[i] @ Bm[axd[i]] @ xt[i]) / abs(t2d[i]) for i in range(3)) < 1e-12)
# ---------------- generator: 1R : m z, CRN across model/iso ----------------
def generate(N, m, stream_ids, S_list, chunk_clusters=2000):
    """returns per-system dict(T1, AX, T2) plus cluster ids; same (R, z) for every system in S_list (CRN)."""
    K = N // m; assert K * m == N; rr = rng_for('rotation', *stream_ids); rz = rng_for('gaussian', *stream_ids); out = [dict(T1=np.empty(N), AX=np.empty(N, np.int32), T2=np.empty(N)) for _ in S_list]; cid = np.repeat(np.arange(K), m); t_rot = 0.0
    for k0 in range(0, K, chunk_clusters):
        k1 = min(K, k0 + chunk_clusters); t = time.perf_counter(); Rs = Rotation.random(num=k1 - k0, rng=rr).as_matrix(); Ds = D_batch(Rs); t_rot += time.perf_counter() - t
        Z = rz.standard_normal((k1 - k0, m, 21))
        for s, S in enumerate(S_list):
            X = np.einsum('kab,kmb->kma', Ds @ S, Z).reshape(-1, 21); T1, AX, T2 = scan(X); sl = slice(k0 * m, k1 * m); out[s]['T1'][sl], out[s]['AX'][sl], out[s]['T2'][sl] = T1, AX, T2
    return out, cid, dict(K=K, m=m, rotation_seconds=t_rot)
# ---------------- calibration sample (isotropic, stream calibration) -> E_sel band ----------------
(cal,), cal_cid, cal_info = generate(CFG['N_CAL'], 100, (0,), [S_I]); q16, q84 = np.quantile(cal['T2'], [0.16, 0.84]); p1_obs = float(np.mean(cal['T1'] <= T1o))
REC['calibration'] = dict(N=CFG['N_CAL'], m=100, q16=float(q16), q84=float(q84), T1_obs=T1o, T2_obs=T2o, p1_obs=p1_obs, OBS_SMINUS_NORMAL=bool(q16 <= T2o <= q84), T1_med=float(np.median(cal['T1'])), T2_med=float(np.median(cal['T2'])), P_T1_le_obs=p1_obs)
print(f'calibration: T1_med {np.median(cal["T1"]):.1f} T2 band [{q16:.1f},{q84:.1f}] p1_obs {p1_obs:.4f} ({time.time()-T0:.0f}s)'); sys.stdout.flush()
# map-free isotropic engine vs the A5 map-based selection-adjusted null (l2_4 band, 1000 synalm maps): medians must lie inside the A5 bootstrap 95% CI
zA5 = np.load(os.path.join(MT, 'results/step1_phaseA/A5_freeze/a5_null_selection.npz')); a5t1, a5t2 = zA5['T1_l2_4'], zA5['T2_l2_4']; rb5 = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 10, 9]))
ci1 = np.quantile([np.median(rb5.choice(a5t1, len(a5t1))) for _ in range(2000)], [0.025, 0.975]); ci2 = np.quantile([np.median(rb5.choice(a5t2, len(a5t2))) for _ in range(2000)], [0.025, 0.975])
GATES['G_iso_engine_matches_A5_null'] = bool(ci1[0] <= np.median(cal['T1']) <= ci1[1] and ci2[0] <= np.median(cal['T2']) <= ci2[1])
REC['a5_null_crosscheck'] = dict(A5_T1_med=float(np.median(a5t1)), A5_T1_med_CI=ci1.tolist(), A5_T2_med=float(np.median(a5t2)), A5_T2_med_CI=ci2.tolist(), A5_p1=float(np.mean(a5t1 <= T1o)), engine_T1_med=float(np.median(cal['T1'])), engine_T2_med=float(np.median(cal['T2'])), engine_p1=p1_obs)
def esel(d, t1=T1o): return (d['T1'] <= t1) & (d['T2'] >= q16) & (d['T2'] <= q84)
# ---------------- A10-2: m sensitivity ----------------
def cluster_boot(hM, hI, K, B, seed):
    """paired cluster bootstrap of Q = sum(hM)/sum(hI) with per-cluster hit counts hM, hI (K,)."""
    rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 10, STREAM['bootstrap'], seed])); idx = rb.integers(0, K, (B, K)); q = []
    for b in range(B):
        c = np.bincount(idx[b], minlength=K); sm, si = c @ hM, c @ hI; q.append(sm / si if si > 0 else np.inf)
    q = np.array(q); return q
MS = {}; KEEP = {}
def analyse(dM, dI, cid, K, m, rep):
    eM, eI = esel(dM), esel(dI); hM, hI = np.bincount(cid, weights=eM, minlength=K), np.bincount(cid, weights=eI, minlength=K); PM, PI = eM.mean(), eI.mean(); Q = PM / PI if PI > 0 else np.inf
    qs = [cluster_boot(hM, hI, K, CFG['B_BOOT'], 100 * m + 10 * rep + s) for s in range(5)]; cis = [np.quantile(q[np.isfinite(q)], [0.025, 0.975]) if np.isfinite(q).sum() > 10 else np.array([np.nan, np.nan]) for q in qs]
    widths = np.array([c[1] - c[0] for c in cis]); ci = cis[0]; rel_half = float((ci[1] - ci[0]) / 2 / Q) if np.isfinite(Q) and Q > 0 else np.inf; se_log = float(np.std(np.log(qs[0][np.isfinite(qs[0]) & (qs[0] > 0)])))
    return dict(K=K, N=CFG['N'], P_M=float(PM), P_I=float(PI), hits_M=int(eM.sum()), hits_I=int(eI.sum()), Q=float(Q), logQ=float(np.log(Q)) if np.isfinite(Q) and Q > 0 else None, se_logQ_bootstrap=se_log, CI95=[float(ci[0]), float(ci[1])], CI_rel_halfwidth=rel_half,
                CI_width_CV_5seeds=float(widths.std() / widths.mean()) if np.all(np.isfinite(widths)) else None, event_positive_clusters_M=int((hM > 0).sum()), event_positive_clusters_I=int((hI > 0).sum()),
                precision_gate=bool((hM > 0).sum() >= 50 and (hI > 0).sum() >= 50 and rel_half <= 0.20 and np.all(np.isfinite(widths)) and widths.std() / widths.mean() < 0.2),
                P_M_T1_le_obs=float(np.mean(dM['T1'] <= T1o)), P_I_T1_le_obs=float(np.mean(dI['T1'] <= T1o)), T1_med_M=float(np.median(dM['T1'])), T1_med_I=float(np.median(dI['T1'])), T2_med_M=float(np.median(dM['T2'])), T2_med_I=float(np.median(dI['T2']))), hM, hI
for m in M_LIST:
    for rep in (1, 2):                                                                     # two independent replicates per m (fresh rotation/gaussian streams)
        t = time.time(); (dM, dI), cid, ginfo = generate(CFG['N'], m, (m, rep), [S_M, S_I]); K = ginfo['K']; r, hM, hI = analyse(dM, dI, cid, K, m, rep); r.update(rotation_seconds=ginfo['rotation_seconds'], seconds=time.time() - t); MS[f'm{m}_rep{rep}'] = r
        if rep == 1: KEEP[m] = dict(dM=dM, dI=dI, cid=cid, K=K, hM=hM, hI=hI)
        print(f'm={m} rep{rep}: K={K} P_M={r["P_M"]:.5f} P_I={r["P_I"]:.5f} Q={r["Q"]:.3f} CI={np.round(r["CI95"],3).tolist()} relhalf={r["CI_rel_halfwidth"]:.3f} ev+ M/I={r["event_positive_clusters_M"]}/{r["event_positive_clusters_I"]} rot {ginfo["rotation_seconds"]:.0f}s total {time.time()-t:.0f}s'); sys.stdout.flush()
def lq(k): return MS[k]['logQ']
def se(k): return MS[k]['se_logQ_bootstrap']
d_cross = abs(lq('m100_rep1') - lq('m10_rep1')); d_rep10 = abs(lq('m10_rep2') - lq('m10_rep1')); d_rep100 = abs(lq('m100_rep2') - lq('m100_rep1'))
hw_diff = 1.959964 * np.sqrt(se('m10_rep1') ** 2 + se('m100_rep1') ** 2); hw_single10 = (np.log(MS['m10_rep1']['CI95'][1]) - np.log(MS['m10_rep1']['CI95'][0])) / 2
M_DECISION = dict(delta_logQ_m100_vs_m10=float(d_cross), replicate_delta_logQ_m10=float(d_rep10), replicate_delta_logQ_m100=float(d_rep100),
                  difference_CI_halfwidth=float(hw_diff), single_CI_halfwidth_m10=float(hw_single10),
                  criterion_difference_CI=bool(d_cross <= hw_diff), criterion_single_CI_m10=bool(d_cross <= hw_single10),
                  m100_precision_gate=all(MS[f'm100_rep{r}']['precision_gate'] for r in (1, 2)), m10_precision_gate=all(MS[f'm10_rep{r}']['precision_gate'] for r in (1, 2)),
                  adopted_m=(100 if (d_cross <= hw_diff and all(MS[f'm100_rep{r}']['precision_gate'] for r in (1, 2))) else 10),
                  rule='adopt m=100 iff |logQ(m=100) - logQ(m=10)| <= 1.96*sqrt(se10^2 + se100^2) (bootstrap SEs of two INDEPENDENT estimates) AND both m=100 replicates pass the precision gate; else m=10. '
                       'The single-estimate-CI criterion of the design note v1.0 is recorded for transparency (it under-covers a difference of two independent estimates).')
print('m decision:', M_DECISION); sys.stdout.flush()
# ---------------- A10-3: calibration pathway (no re-scan) ----------------
(dP,), _, _ = generate(CFG['N_PSEUDO'], 1, (0,), [S_I]); pseudo_T1 = dP['T1']; pseudo_T2 = dP['T2']
m_adopt = M_DECISION['adopted_m']; dM, dI, cid, K = KEEP[m_adopt]['dM'], KEEP[m_adopt]['dI'], KEEP[m_adopt]['cid'], KEEP[m_adopt]['K']; bandM, bandI = (dM['T2'] >= q16) & (dM['T2'] <= q84), (dI['T2'] >= q16) & (dI['T2'] <= q84)
oM = np.argsort(dM['T1']); oI = np.argsort(dI['T1']); T1M_s, T1I_s = dM['T1'][oM], dI['T1'][oI]; cidM_s, cidI_s = cid[oM], cid[oI]; bandM_s, bandI_s = bandM[oM], bandI[oI]
# per-cluster hit tables for every pseudo threshold: H[k, p] = #{i in cluster k: T1_i <= t_p and band}
def hit_table(T1s, band_s, cid_s, thresholds):
    H = np.zeros((K, len(thresholds))); cum = np.cumsum(band_s.astype(np.int64))
    for p, t in enumerate(thresholds):
        n = np.searchsorted(T1s, t, side='right'); H[:, p] = np.bincount(cid_s[:n][band_s[:n]], minlength=K)
    return H
t = time.time(); HM, HI = hit_table(T1M_s, bandM_s, cidM_s, pseudo_T1), hit_table(T1I_s, bandI_s, cidI_s, pseudo_T1)
PMp, PIp = HM.sum(0) / len(dM['T1']), HI.sum(0) / len(dI['T1']); Qp = np.where(PIp > 0, PMp / np.maximum(PIp, 1e-300), np.inf)
rb = np.random.default_rng(np.random.SeedSequence([MASTER_SEED, 10, STREAM['bootstrap'], 777])); Wc = np.zeros((CFG['B_BOOT'], K))
for b in range(CFG['B_BOOT']): Wc[b] = np.bincount(rb.integers(0, K, K), minlength=K)
SM, SI = Wc @ HM, Wc @ HI; Qb = np.where(SI > 0, SM / np.maximum(SI, 1e-300), np.inf); lo = np.array([np.quantile(Qb[:, p][np.isfinite(Qb[:, p])], 0.025) if np.isfinite(Qb[:, p]).sum() > 10 else np.nan for p in range(len(pseudo_T1))])
# density ratio D at pseudo T_obs (secondary): 2D Gaussian KDE on a FITTING subsample (separate from evaluation)
rf = rng_for('calibration', 5); fM = rf.choice(len(dM['T1']), CFG['N_FIT'], replace=False); fI = rf.choice(len(dI['T1']), CFG['N_FIT'], replace=False)
kM = gaussian_kde(np.vstack([dM['T1'][fM], dM['T2'][fM]])); kI = gaussian_kde(np.vstack([dI['T1'][fI], dI['T2'][fI]])); Dp = kM(np.vstack([pseudo_T1, pseudo_T2])) / np.maximum(kI(np.vstack([pseudo_T1, pseudo_T2])), 1e-300)
support = (lo >= 3) & (np.log(Dp) > 0); strong_raw = (lo >= 10) & (Dp > 1)   # strong also needs D lower-CI > 1 and matched/native agreement (not available for one system) -> reported as raw
n_p = len(pseudo_T1); k_s = int(support.sum()); k_st = int(strong_raw.sum())
def wilson_upper(k, n, zq=1.959964):
    p = k / n; return float((p + zq * zq / (2 * n) + zq * np.sqrt(p * (1 - p) / n + zq * zq / (4 * n * n))) / (1 + zq * zq / n))
CAL = dict(n_pseudo=n_p, pseudo_stream='pseudo (independent isotropic draws, m=1)', method='T1-threshold indicator re-evaluation on sorted T1 + per-cluster hit tables; ONE shared cluster-resampling matrix (B x K) for all pseudo; Q lower CI = 2.5% percentile',
           FWFSR_support=k_s / n_p, FWFSR_support_wilson_upper=wilson_upper(k_s, n_p), FWFSR_strong_raw=k_st / n_p, FWFSR_strong_raw_wilson_upper=wilson_upper(k_st, n_p),
           pseudo_T1_quantiles=np.quantile(pseudo_T1, [0.05, 0.5, 0.95]).tolist(), Q_pseudo_median=float(np.median(Qp[np.isfinite(Qp)])) if np.isfinite(Qp).any() else None, seconds=time.time() - t,
           note='pathway verification on ONE model point (E7_b1_A) and one system; the true familywise calibration integrates all families and both systems at official n_pseudo >= 2000')
print('calibration pathway:', {k: v for k, v in CAL.items() if k not in ('method', 'note')}); sys.stdout.flush()
# ---------------- A10-1: W2 on real engine output (whitened by calibration sample), exact 2D, cluster-wise subsampling ----------------
import ot
mu_c = np.array([cal['T1'].mean(), cal['T2'].mean()]); Sig_c = np.cov(np.vstack([cal['T1'], cal['T2']])); Sih = np.linalg.inv(np.linalg.cholesky(Sig_c))
def white(d): return (np.column_stack([d['T1'], d['T2']]) - mu_c) @ Sih.T
def sub_clusters(Tw, cid, K, n_sub, rng, m):
    kk = rng.choice(K, n_sub // m, replace=False); sel = np.isin(cid, kk); return Tw[sel]
def w2_exact(a, b):
    Mc = ot.dist(a, b, metric='sqeuclidean'); v, log = ot.emd2(np.full(len(a), 1 / len(a)), np.full(len(b), 1 / len(b)), Mc, numItermax=1_000_000, log=True); return float(np.sqrt(v)), log
TwM, TwI = white(dM), white(dI); rw = rng_for('w2', 1); t = time.time()
(dI2,), cid2, _ = generate(CFG['N_SUB_W2'] * 4, m_adopt, (2,), [S_I]); TwI2 = white(dI2); K2 = CFG['N_SUB_W2'] * 4 // m_adopt   # second independent isotropic sample for the null
null = []; warn = 0
for b in range(CFG['B_NULL']):
    a = sub_clusters(TwI, cid, K, CFG['N_SUB_W2'], rw, m_adopt); c = sub_clusters(TwI2, cid2, K2, CFG['N_SUB_W2'], rw, m_adopt); v, lg = w2_exact(a, c); null.append(v); warn += int(bool(lg.get('warning')))
null = np.array(null); q99 = float(np.quantile(null, 0.99)); vMI, _ = w2_exact(sub_clusters(TwM, cid, K, CFG['N_SUB_W2'], rw, m_adopt), sub_clusters(TwI, cid, K, CFG['N_SUB_W2'], rw, m_adopt))
W2 = dict(estimator='exact 2D W2 (POT emd2, sqeuclidean, sqrt)', pot_version=ot.__version__, n_sub=CFG['N_SUB_W2'], subsampling='cluster-wise (1R:m z clusters kept whole)', m=m_adopt, B_null=CFG['B_NULL'], null_q99=q99, null_median=float(np.median(null)), null_max=float(null.max()), emd_warnings=warn,
          W2_model_vs_iso=vMI, model_vs_iso_exceeds_q99=bool(vMI > q99), whitening=dict(mu=mu_c.tolist(), Sigma=Sig_c.tolist(), source='calibration sample'), seconds=time.time() - t)
GATES['G_w2_null_finite'] = bool(np.all(np.isfinite(null)) and warn == 0 and np.isfinite(vMI))
print('W2:', {k: v for k, v in W2.items() if k != 'whitening'}); sys.stdout.flush()
# ---------------- provenance ----------------
REQUIRED = ['G_bstack_file_sha', 'G_bstack_array_sha', 'G_cov_manifest_target', 'G_cov_file_sha', 'G_cov_array_sha', 'G_bridge_sha', 'G_cvec_sha', 'G_basis_order', 'G_step0_obs_bound', 'G_matched_power', 'G_sqrt_hard', 'G_quadrature_orthonormal',
            'G_D_batch_matches_single', 'G_D_orthogonal', 'G_D_homomorphism', 'G_scan_vs_direct', 'G_iso_engine_matches_A5_null', 'G_w2_null_finite']
GATES['G_gate_inventory_exact'] = (set(GATES) | {'G_gate_inventory_exact'} == set(REQUIRED) | {'G_gate_inventory_exact'}); REQUIRED.append('G_gate_inventory_exact')
status = ('A10_VALID' if MODE == 'official' else 'SMOKE_PASS') if all(GATES[k] for k in REQUIRED) else 'FAILED'
prov = dict(notebook=f'Step1 Phase A-10 W2 / m-sensitivity / calibration pathway v1.0 [{MODE}]', mt_commit=os.environ.get('A10_MT_COMMIT'), notebook_identity=json.loads(os.environ.get('A10_NB_IDENTITY', '{}')), status=status, timestamp=time.strftime('%Y-%m-%dT%H:%M:%S'), config=CFG, master_seed=MASTER_SEED, streams=STREAM,
            environment=dict(python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, healpy=hp.__version__, pandas=pd.__version__, pot=ot.__version__, platform=platform.platform()),
            gates=GATES, required_gates=REQUIRED, assets=dict(bstack=os.path.basename(BST), cov=os.path.basename(COVF), cov_tag=covman['tag'], step0_frozen=os.path.basename(S0F), step0_csv=os.path.basename(S0CSV)), covariance=REC['covariance'], calibration=REC['calibration'], a5_null_crosscheck=REC['a5_null_crosscheck'],
            m_sensitivity=MS, m_decision=M_DECISION, calibration_pathway=CAL, w2=W2, total_seconds=time.time() - T0)
json.dump(prov, open(os.path.join(OUT, 'a10_provenance.json'), 'w'), indent=1, ensure_ascii=False); np.savez_compressed(os.path.join(OUT, 'a10_vectors.npz'), cal_T1=cal['T1'], cal_T2=cal['T2'], pseudo_T1=pseudo_T1, pseudo_T2=pseudo_T2, w2_null=null, pseudo_Q=Qp, pseudo_Q_lowerCI=lo, pseudo_D=Dp, **{f'm{m}_rep1_h{s}': KEEP[m][f'h{s}'] for m in M_LIST for s in ('M', 'I')})
print('STATUS =', status, '| gates', sum(GATES.values()), '/', len(GATES), '| total %.0fs' % (time.time() - T0))


## 実行手順
1. **smoke**（fresh runtime・約 1 分）：先頭に `A10_MODE='smoke'` セルを追加して Run all → `STATUS = SMOKE_PASS`（19/19）。
2. **official**（fresh runtime・CPU で可・約 20–40 分）：純正ノートブックを Run all → `STATUS = A10_VALID`。
3. 返送：`runs_step1_phaseA/a10_v1.0_{smoke,official}/` の `a10_provenance.json`・`a10_vectors.npz` と全セル出力。
Phase A 調査ノートのため commit は任意（Phase A の他ノートと同様，結果は設計ノート v1.1 と rules v1.0 draft に転記する）。
